In [1]:
!pip install optuna-integration[sklearn]

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score,ParameterGrid
from sklearn.linear_model import  LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler ,MinMaxScaler
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from optuna.integration import OptunaSearchCV
from optuna.distributions import IntDistribution, FloatDistribution, CategoricalDistribution


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df=pd.read_csv('New Dataset/spain.csv')
df.head()


,Y,X,data_payload_id,instance_datetime,url,agency,platform_type,platform_id,platform_name,gaw_id,...,daily_utc_begin,daily_utc_end,daily_utc_mean,daily_nobs,daily_mmu,daily_columnso2,latest_observation,country,scientific_authority,version
0,28.46,-16.25,2306481,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,9.15,10.52,9.60,4.0,2.181,-0.5,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
1,28.46,-16.25,2306502,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,12.50,17.88,15.22,12.0,1.421,-1.4,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
2,28.46,-16.25,2306482,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,8.85,14.03,10.80,16.0,1.638,-0.9,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
3,28.46,-16.25,2306480,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,12.83,17.27,15.41,10.0,1.602,-0.6,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0
4,28.46,-16.25,2306483,2025/03/05 00:00:00+00,https://woudc.org/archive/Archive-NewFormat/To...,AEMET,STN,401,Santa Cruz (Tenerife),SCO,...,9.55,17.55,15.04,19.0,1.680,-0.8,0,ESP,MIGUEL HERNANDEZ MARTINEZ DE LA PEÑA,0.0


In [3]:
df=df.drop(['Y', 'X', 'data_payload_id', 'instance_datetime', 'url', 'agency',
       'platform_type', 'platform_id', 'platform_name', 'gaw_id',
       'instrument_name', 'instrument_model', 'instrument_number',
       'monthly_date', 'monthly_stddevo3', 'monthly_npts','daily_stddevo3',
        'daily_wlcode', 'daily_obscode', 'monthly_columno3',
        'daily_utc_begin', 'daily_utc_end', 'daily_utc_mean',
       'daily_nobs', 'daily_mmu', 'daily_columnso2', 'latest_observation',
       'country', 'scientific_authority', 'version'], axis = 1)
df.head()

,daily_date,daily_columno3
0,2025-03-06,280.5
1,2025-03-27,310.2
2,2025-03-07,306.3
3,2025-03-05,319.4
4,2025-03-08,313.9


In [4]:
df=df.drop_duplicates()
df=df.dropna()
df.head()


,daily_date,daily_columno3
0,2025-03-06,280.5
1,2025-03-27,310.2
2,2025-03-07,306.3
3,2025-03-05,319.4
4,2025-03-08,313.9


In [5]:
print(f"Date Range: {df.loc[:,'daily_date'][len(df)-1]} to {df.loc[:,'daily_date'][0]}")

Date Range: 2001-03-10 to 2025-03-06


**# Trial 1** <br>
***TrainTestSplit + lag_60 + rolling_avg_7 + exp_avg_7*** <br>


In [6]:
# 2. Sort the dataframe by the full date (oldest to latest)
df1=df.copy()
df1 = df1.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df1['rolling_avg'] = df1['daily_columno3'].shift(1).rolling(window=7).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day
df1['ema_avg'] = df1['daily_columno3'].shift(1).ewm(span=7, adjust=False).mean()


for i in range(1, 61):
    df1[f'lag{i}'] = df1['daily_columno3'].shift(i)


df1.dropna(inplace=True)
print(df1.shape)
df1.head()

(65450, 64)


,daily_date,daily_columno3,rolling_avg,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
60,1980-02-26,340.0,291.528571,290.176070,286.0,287.4,297.2,311.8,294.2,267.9,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
61,1980-02-27,308.0,297.785714,302.632053,340.0,286.0,287.4,297.2,311.8,294.2,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
62,1980-03-05,424.0,303.514286,303.974039,308.0,340.0,286.0,287.4,297.2,311.8,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
63,1980-03-06,323.0,322.057143,333.980530,424.0,308.0,340.0,286.0,287.4,297.2,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,323.657143,331.235397,323.0,424.0,308.0,340.0,286.0,287.4,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [7]:
df1.tail()

,daily_date,daily_columno3,rolling_avg,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
65505,2025-03-31,304.2,318.200000,321.528513,337.5,323.5,296.8,314.8,317.0,310.6,...,331.3,432.5,376.8,415.4,419.5,321.6,411.2,300.6,403.9,309.3
65506,2025-03-31,326.3,314.914286,317.196385,304.2,337.5,323.5,296.8,314.8,317.0,...,399.3,331.3,432.5,376.8,415.4,419.5,321.6,411.2,300.6,403.9
65507,2025-03-31,332.6,317.157143,319.472289,326.3,304.2,337.5,323.5,296.8,314.8,...,351.0,399.3,331.3,432.5,376.8,415.4,419.5,321.6,411.2,300.6
65508,2025-03-31,291.9,319.385714,322.754217,332.6,326.3,304.2,337.5,323.5,296.8,...,308.4,351.0,399.3,331.3,432.5,376.8,415.4,419.5,321.6,411.2
65509,2025-03-31,325.0,316.114286,315.040662,291.9,332.6,326.3,304.2,337.5,323.5,...,368.1,308.4,351.0,399.3,331.3,432.5,376.8,415.4,419.5,321.6


In [8]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_avg')
lag_features.append('ema_avg')
X = df1[lag_features]
y = df1['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002116 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 52360, number of used features: 62
[LightGBM] [Info] Start training from score -0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.648
MAE : 0.472
R2  : 52.71 %

----- Decision Tree -----
RMSE: 0.916
MAE : 0.665
R2  : 5.64 %

----- Random Forest -----
RMSE: 0.648
MAE : 0.472
R2  : 52.80 %

----- Gradient Boosting -----
RMSE: 0.648
MAE : 0.472
R2  : 52.80 %

----- Support Vector Regressor -----
RMSE: 0.668
MAE : 0.482
R2  : 49.82 %

----- XGBoost -----
RMSE: 0.658
MAE : 0.475
R2  : 51.33 %

----- LightGBM -----
RMSE: 0.648
MAE : 0.472
R2  : 52.74 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**Trial 1: Tuning**

In [ ]:

# Ensure data type
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

# Define models with GPU enabled where possible
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(tree_method='gpu_hist', predictor='gpu_predictor', random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(device='gpu', random_state=42)
}

# Define hyperparameter grids

param_grids = {
    'Linear Regression': {},
    'Decision Tree': {
        'max_depth': IntDistribution(5, 15),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Random Forest': {
        'n_estimators': IntDistribution(100, 150),
        'max_depth': IntDistribution(10, 20),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Gradient Boosting': {
        'n_estimators': IntDistribution(100, 150),
        'learning_rate': FloatDistribution(0.01, 0.1),
        'max_depth': IntDistribution(3, 5),
        'min_samples_split': IntDistribution(2, 5)
    },
    'Support Vector Regressor': {
        'C': FloatDistribution(0.1, 10),
        'kernel': CategoricalDistribution(['rbf', 'linear']),
        'epsilon': FloatDistribution(0.01, 0.1)
    },
    'XGBoost': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 7),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'gamma': FloatDistribution(0, 1),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(1, 2)
    },
    'LightGBM': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 5),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'num_leaves': IntDistribution(15, 31),
        'min_child_samples': IntDistribution(10, 20),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(0, 1)
    }
}

# Models to tune with OptunaSearchCV
models_to_optimize = {'Random Forest', 'Gradient Boosting', 'Support Vector Regressor', 'XGBoost', 'LightGBM'}

best_models = {}
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    if param_grids[name]:
        # Use OptunaSearchCV for tuning (Bayesian optimization)
        n_trials = 20
        search = OptunaSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            n_trials=n_trials,
            cv=3,
            scoring='r2',
            n_jobs=2,
            random_state=42,
            verbose=1
        )
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        print(f"{name} Best Parameters: {search.best_params_}")

    else:
        model.fit(X_train, y_train)
        best_models[name] = model

    # Predict and evaluate
    preds = best_models[name].predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'Model': best_models[name],
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

print("\nModel Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")

/tmp/ipykernel_294262/373749671.py:78: ExperimentalWarning: OptunaSearchCV is experimental (supported from v0.17.0). The interface can change in the future.
  search = OptunaSearchCV(
[I 2025-06-01 22:07:17,604] A new study created in memory with name: no-name-8a93711d-aab4-460b-9610-26711cc3754e



Training Linear Regression...

Training Decision Tree...


[I 2025-06-01 22:07:21,196] Trial 0 finished with value: 0.31332908955819755 and parameters: {'max_depth': 13, 'min_samples_split': 3, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.31332908955819755.
[I 2025-06-01 22:07:21,199] Trial 1 finished with value: 0.3100954645680665 and parameters: {'max_depth': 13, 'min_samples_split': 2, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.31332908955819755.
[I 2025-06-01 22:07:23,319] Trial 2 finished with value: 0.43692041801761755 and parameters: {'max_depth': 8, 'min_samples_split': 3, 'min_samples_leaf': 1}. Best is trial 2 with value: 0.43692041801761755.
[I 2025-06-01 22:07:24,593] Trial 4 finished with value: 0.4779388949399535 and parameters: {'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 1}. Best is trial 4 with value: 0.4779388949399535.
[I 2025-06-01 22:07:25,288] Trial 3 finished with value: 0.2722890181716229 and parameters: {'max_depth': 15, 'min_samples_split': 5, 'min_samples_leaf': 1}. Best is trial 4

Decision Tree Best Parameters: {'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 1}

Training Random Forest...


[I 2025-06-01 22:12:02,036] Trial 1 finished with value: 0.49349183296072874 and parameters: {'n_estimators': 113, 'max_depth': 11, 'min_samples_split': 2, 'min_samples_leaf': 1}. Best is trial 1 with value: 0.49349183296072874.
[I 2025-06-01 22:14:05,914] Trial 0 finished with value: 0.49249778574764713 and parameters: {'n_estimators': 123, 'max_depth': 15, 'min_samples_split': 3, 'min_samples_leaf': 1}. Best is trial 1 with value: 0.49349183296072874.
[I 2025-06-01 22:18:01,526] Trial 2 finished with value: 0.49311915441621257 and parameters: {'n_estimators': 124, 'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 2}. Best is trial 1 with value: 0.49349183296072874.
[I 2025-06-01 22:19:05,579] Trial 3 finished with value: 0.494282547905847 and parameters: {'n_estimators': 146, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 2}. Best is trial 3 with value: 0.494282547905847.
[I 2025-06-01 22:22:12,049] Trial 4 finished with value: 0.49310101637637893 and paramet

Random Forest Best Parameters: {'n_estimators': 135, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 2}

Training Gradient Boosting...


***# Trial 2 : TrainTestSplit + lag_30 + rolling_std_3 + exp_avg_3***

In [ ]:
# 2. Sort the dataframe by the full date (oldest to latest)
df2=df.copy()
df2 = df2.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df2['rolling_std'] = df2['daily_columno3'].shift(1).rolling(window=3).std()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day
df2['ema_avg'] = df2['daily_columno3'].shift(1).ewm(span=3, adjust=False).mean()


for i in range(1, 31):
    df2[f'lag{i}'] = df2['daily_columno3'].shift(i)


df2.dropna(inplace=True)
print(df2.shape)
df2.head()

(65480, 34)


,daily_date,daily_columno3,rolling_std,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag21,lag22,lag23,lag24,lag25,lag26,lag27,lag28,lag29,lag30
30,1976-09-07,273.8,4.409460,287.181151,289.4,280.6,284.5,290.3,295.2,303.0,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
31,1976-09-08,284.5,7.821338,280.490575,273.8,289.4,280.6,284.5,290.3,295.2,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
32,1976-09-09,283.5,7.977677,282.495288,284.5,273.8,289.4,280.6,284.5,290.3,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
33,1976-09-10,283.5,5.910161,282.997644,283.5,284.5,273.8,289.4,280.6,284.5,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
34,1976-09-13,307.9,0.577350,283.248822,283.5,283.5,284.5,273.8,289.4,280.6,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [ ]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 31)]
lag_features.append('rolling_std')
lag_features.append('ema_avg')
X = df2[lag_features]
y = df2['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001348 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8160
[LightGBM] [Info] Number of data points in the train set: 52384, number of used features: 32
[LightGBM] [Info] Start training from score 0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.646
MAE : 0.470
R2  : 52.97 %

----- Decision Tree -----
RMSE: 0.927
MAE : 0.672
R2  : 3.37 %

----- Random Forest -----
RMSE: 0.650
MAE : 0.473
R2  : 52.51 %

----- Gradient Boosting -----
RMSE: 0.649
MAE : 0.472
R2  : 52.56 %

----- Support Vector Regressor -----
RMSE: 0.664
MAE : 0.477
R2  : 50.36 %

----- XGBoost -----
RMSE: 0.667
MAE : 0.481
R2  : 49.95 %

----- LightGBM -----
RMSE: 0.649
MAE : 0.472
R2  : 52.62 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***Tuning 2***

In [ ]:

# Ensure data type
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

# Define models with GPU enabled where possible
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(tree_method='gpu_hist', predictor='gpu_predictor', random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(device='gpu', random_state=42)
}

# Define hyperparameter grids

param_grids = {
    'Linear Regression': {},
    'Decision Tree': {
        'max_depth': IntDistribution(5, 15),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Random Forest': {
        'n_estimators': IntDistribution(100, 150),
        'max_depth': IntDistribution(10, 20),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Gradient Boosting': {
        'n_estimators': IntDistribution(100, 150),
        'learning_rate': FloatDistribution(0.01, 0.1),
        'max_depth': IntDistribution(3, 5),
        'min_samples_split': IntDistribution(2, 5)
    },
    'Support Vector Regressor': {
        'C': FloatDistribution(0.1, 10),
        'kernel': CategoricalDistribution(['rbf', 'linear']),
        'epsilon': FloatDistribution(0.01, 0.1)
    },
    'XGBoost': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 7),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'gamma': FloatDistribution(0, 1),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(1, 2)
    },
    'LightGBM': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 5),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'num_leaves': IntDistribution(15, 31),
        'min_child_samples': IntDistribution(10, 20),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(0, 1)
    }
}

# Models to tune with OptunaSearchCV
models_to_optimize = {'Random Forest', 'Gradient Boosting', 'Support Vector Regressor', 'XGBoost', 'LightGBM'}

best_models = {}
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    if param_grids[name]:
        # Use OptunaSearchCV for tuning (Bayesian optimization)
        n_trials = 20
        search = OptunaSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            n_trials=n_trials,
            cv=3,
            scoring='r2',
            n_jobs=2,
            random_state=42,
            verbose=1
        )
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        print(f"{name} Best Parameters: {search.best_params_}")

    else:
        model.fit(X_train, y_train)
        best_models[name] = model

    # Predict and evaluate
    preds = best_models[name].predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'Model': best_models[name],
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

print("\nModel Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")

**Trial 3 : TrainTestSplit + lag_80 + rolling_avg_14 + rolling_std_14 + exp_avg_14**

In [ ]:
# 2. Sort the dataframe by the full date (oldest to latest)
df3=df.copy()
df3 = df3.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df3['rolling_std'] = df3['daily_columno3'].shift(1).rolling(window=14).std()
df3['rolling_avg'] = df3['daily_columno3'].shift(1).rolling(window=14).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day
df3['ema_avg'] = df3['daily_columno3'].shift(1).ewm(span=14, adjust=False).mean()


for i in range(1, 81):
    df3[f'lag{i}'] = df3['daily_columno3'].shift(i)


df3.dropna(inplace=True)
print(df3.shape)
df3.head()

(65430, 85)


,daily_date,daily_columno3,rolling_std,rolling_avg,ema_avg,lag1,lag2,lag3,lag4,lag5,...,lag71,lag72,lag73,lag74,lag75,lag76,lag77,lag78,lag79,lag80
80,1980-04-28,373.0,21.702914,360.642857,356.398440,354.0,347.0,344.0,360.0,386.0,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
81,1980-04-30,413.0,21.225762,360.071429,358.611981,373.0,354.0,347.0,344.0,360.0,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
82,1980-05-12,389.0,23.968500,365.785714,365.863717,413.0,373.0,354.0,347.0,344.0,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
83,1980-05-13,373.0,24.464821,368.285714,368.948555,389.0,413.0,373.0,354.0,347.0,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
84,1980-05-14,376.0,21.699749,371.428571,369.488748,373.0,389.0,413.0,373.0,354.0,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [ ]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 81)]
lag_features.append('rolling_avg')
lag_features.append('rolling_std')
lag_features.append('ema_avg')
X = df3[lag_features]
y = df3['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003960 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 21165
[LightGBM] [Info] Number of data points in the train set: 52344, number of used features: 83
[LightGBM] [Info] Start training from score 0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.648
MAE : 0.472
R2  : 52.72 %

----- Decision Tree -----
RMSE: 0.907
MAE : 0.661
R2  : 7.51 %

----- Random Forest -----
RMSE: 0.649
MAE : 0.474
R2  : 52.69 %

----- Gradient Boosting -----
RMSE: 0.648
MAE : 0.473
R2  : 52.77 %

----- Support Vector Regressor -----
RMSE: 0.672
MAE : 0.485
R2  : 49.28 %

----- XGBoost -----
RMSE: 0.662
MAE : 0.480
R2  : 50.66 %

----- LightGBM -----
RMSE: 0.647
MAE : 0.472
R2  : 52.93 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***Tuning 3***

In [ ]:

# Ensure data type
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

# Define models with GPU enabled where possible
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(tree_method='gpu_hist', predictor='gpu_predictor', random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(device='gpu', random_state=42)
}

# Define hyperparameter grids

param_grids = {
    'Linear Regression': {},
    'Decision Tree': {
        'max_depth': IntDistribution(5, 15),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Random Forest': {
        'n_estimators': IntDistribution(100, 150),
        'max_depth': IntDistribution(10, 20),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Gradient Boosting': {
        'n_estimators': IntDistribution(100, 150),
        'learning_rate': FloatDistribution(0.01, 0.1),
        'max_depth': IntDistribution(3, 5),
        'min_samples_split': IntDistribution(2, 5)
    },
    'Support Vector Regressor': {
        'C': FloatDistribution(0.1, 10),
        'kernel': CategoricalDistribution(['rbf', 'linear']),
        'epsilon': FloatDistribution(0.01, 0.1)
    },
    'XGBoost': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 7),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'gamma': FloatDistribution(0, 1),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(1, 2)
    },
    'LightGBM': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 5),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'num_leaves': IntDistribution(15, 31),
        'min_child_samples': IntDistribution(10, 20),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(0, 1)
    }
}

# Models to tune with OptunaSearchCV
models_to_optimize = {'Random Forest', 'Gradient Boosting', 'Support Vector Regressor', 'XGBoost', 'LightGBM'}

best_models = {}
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    if param_grids[name]:
        # Use OptunaSearchCV for tuning (Bayesian optimization)
        n_trials = 20
        search = OptunaSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            n_trials=n_trials,
            cv=3,
            scoring='r2',
            n_jobs=2,
            random_state=42,
            verbose=1
        )
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        print(f"{name} Best Parameters: {search.best_params_}")

    else:
        model.fit(X_train, y_train)
        best_models[name] = model

    # Predict and evaluate
    preds = best_models[name].predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'Model': best_models[name],
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

print("\nModel Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")

**Trial 4 : TrainTestSplit + lag_60 + rolling_avg_7**

In [ ]:
# 2. Sort the dataframe by the full date (oldest to latest)
df4=df.copy()
df4 = df4.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df4['rolling_avg'] = df4['daily_columno3'].shift(1).rolling(window=7).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day


for i in range(1, 61):
    df4[f'lag{i}'] = df4['daily_columno3'].shift(i)


df4.dropna(inplace=True)
print(df4.shape)
df4.head()

(65450, 63)


,daily_date,daily_columno3,rolling_avg,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
60,1980-02-26,340.0,291.528571,286.0,287.4,297.2,311.8,294.2,267.9,296.2,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
61,1980-02-27,308.0,297.785714,340.0,286.0,287.4,297.2,311.8,294.2,267.9,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
62,1980-03-05,424.0,303.514286,308.0,340.0,286.0,287.4,297.2,311.8,294.2,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
63,1980-03-06,323.0,322.057143,424.0,308.0,340.0,286.0,287.4,297.2,311.8,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,323.657143,323.0,424.0,308.0,340.0,286.0,287.4,297.2,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [ ]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_avg')
X = df4[lag_features]
y = df4['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003074 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 52360, number of used features: 61
[LightGBM] [Info] Start training from score -0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.648
MAE : 0.472
R2  : 52.69 %

----- Decision Tree -----
RMSE: 0.932
MAE : 0.673
R2  : 2.22 %

----- Random Forest -----
RMSE: 0.649
MAE : 0.474
R2  : 52.64 %

----- Gradient Boosting -----
RMSE: 0.649
MAE : 0.472
R2  : 52.60 %

----- Support Vector Regressor -----
RMSE: 0.668
MAE : 0.482
R2  : 49.80 %

----- XGBoost -----
RMSE: 0.661
MAE : 0.479
R2  : 50.87 %

----- LightGBM -----
RMSE: 0.648
MAE : 0.472
R2  : 52.74 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***Tuning 4***

In [ ]:

# Ensure data type
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

# Define models with GPU enabled where possible
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(tree_method='gpu_hist', predictor='gpu_predictor', random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(device='gpu', random_state=42)
}

# Define hyperparameter grids

param_grids = {
    'Linear Regression': {},
    'Decision Tree': {
        'max_depth': IntDistribution(5, 15),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Random Forest': {
        'n_estimators': IntDistribution(100, 150),
        'max_depth': IntDistribution(10, 20),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Gradient Boosting': {
        'n_estimators': IntDistribution(100, 150),
        'learning_rate': FloatDistribution(0.01, 0.1),
        'max_depth': IntDistribution(3, 5),
        'min_samples_split': IntDistribution(2, 5)
    },
    'Support Vector Regressor': {
        'C': FloatDistribution(0.1, 10),
        'kernel': CategoricalDistribution(['rbf', 'linear']),
        'epsilon': FloatDistribution(0.01, 0.1)
    },
    'XGBoost': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 7),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'gamma': FloatDistribution(0, 1),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(1, 2)
    },
    'LightGBM': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 5),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'num_leaves': IntDistribution(15, 31),
        'min_child_samples': IntDistribution(10, 20),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(0, 1)
    }
}

# Models to tune with OptunaSearchCV
models_to_optimize = {'Random Forest', 'Gradient Boosting', 'Support Vector Regressor', 'XGBoost', 'LightGBM'}

best_models = {}
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    if param_grids[name]:
        # Use OptunaSearchCV for tuning (Bayesian optimization)
        n_trials = 20
        search = OptunaSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            n_trials=n_trials,
            cv=3,
            scoring='r2',
            n_jobs=2,
            random_state=42,
            verbose=1
        )
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        print(f"{name} Best Parameters: {search.best_params_}")

    else:
        model.fit(X_train, y_train)
        best_models[name] = model

    # Predict and evaluate
    preds = best_models[name].predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'Model': best_models[name],
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

print("\nModel Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")

**Trial 5 : TrainTestSplit + lag_60 + rolling_avg_7**

In [ ]:
# 2. Sort the dataframe by the full date (oldest to latest)
df5=df.copy()
df5 = df5.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df5['rolling_avg'] = df5['daily_columno3'].shift(1).rolling(window=7).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day


for i in range(1, 61):
    df5[f'lag{i}'] = df5['daily_columno3'].shift(i)


df5.dropna(inplace=True)
print(df5.shape)
df5.head()

(65450, 63)


,daily_date,daily_columno3,rolling_avg,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
60,1980-02-26,340.0,291.528571,286.0,287.4,297.2,311.8,294.2,267.9,296.2,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
61,1980-02-27,308.0,297.785714,340.0,286.0,287.4,297.2,311.8,294.2,267.9,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
62,1980-03-05,424.0,303.514286,308.0,340.0,286.0,287.4,297.2,311.8,294.2,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
63,1980-03-06,323.0,322.057143,424.0,308.0,340.0,286.0,287.4,297.2,311.8,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,323.657143,323.0,424.0,308.0,340.0,286.0,287.4,297.2,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [ ]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_avg')
X = df5[lag_features]
y = df5['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002998 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 52360, number of used features: 61
[LightGBM] [Info] Start training from score -0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.648
MAE : 0.472
R2  : 52.69 %

----- Decision Tree -----
RMSE: 0.932
MAE : 0.673
R2  : 2.22 %

----- Random Forest -----
RMSE: 0.649
MAE : 0.474
R2  : 52.64 %

----- Gradient Boosting -----
RMSE: 0.649
MAE : 0.472
R2  : 52.60 %

----- Support Vector Regressor -----
RMSE: 0.668
MAE : 0.482
R2  : 49.80 %

----- XGBoost -----
RMSE: 0.661
MAE : 0.479
R2  : 50.87 %

----- LightGBM -----
RMSE: 0.648
MAE : 0.472
R2  : 52.74 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***Tuning 5***

In [ ]:

# Ensure data type
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

# Define models with GPU enabled where possible
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(tree_method='gpu_hist', predictor='gpu_predictor', random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(device='gpu', random_state=42)
}

# Define hyperparameter grids

param_grids = {
    'Linear Regression': {},
    'Decision Tree': {
        'max_depth': IntDistribution(5, 15),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Random Forest': {
        'n_estimators': IntDistribution(100, 150),
        'max_depth': IntDistribution(10, 20),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Gradient Boosting': {
        'n_estimators': IntDistribution(100, 150),
        'learning_rate': FloatDistribution(0.01, 0.1),
        'max_depth': IntDistribution(3, 5),
        'min_samples_split': IntDistribution(2, 5)
    },
    'Support Vector Regressor': {
        'C': FloatDistribution(0.1, 10),
        'kernel': CategoricalDistribution(['rbf', 'linear']),
        'epsilon': FloatDistribution(0.01, 0.1)
    },
    'XGBoost': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 7),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'gamma': FloatDistribution(0, 1),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(1, 2)
    },
    'LightGBM': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 5),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'num_leaves': IntDistribution(15, 31),
        'min_child_samples': IntDistribution(10, 20),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(0, 1)
    }
}

# Models to tune with OptunaSearchCV
models_to_optimize = {'Random Forest', 'Gradient Boosting', 'Support Vector Regressor', 'XGBoost', 'LightGBM'}

best_models = {}
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    if param_grids[name]:
        # Use OptunaSearchCV for tuning (Bayesian optimization)
        n_trials = 20
        search = OptunaSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            n_trials=n_trials,
            cv=3,
            scoring='r2',
            n_jobs=2,
            random_state=42,
            verbose=1
        )
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        print(f"{name} Best Parameters: {search.best_params_}")

    else:
        model.fit(X_train, y_train)
        best_models[name] = model

    # Predict and evaluate
    preds = best_models[name].predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'Model': best_models[name],
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

print("\nModel Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")

**Trial 6 : TrainTestSplit + lag_30 + exp_avg_3**

In [ ]:
# 2. Sort the dataframe by the full date (oldest to latest)
df6=df.copy()
df6 = df6.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df6['ema_avg'] = df6['daily_columno3'].shift(1).ewm(span=3, adjust=False).mean()

# Exponential moving average feature (7-day) - To Avoid data leakage, we shift the column by 1 day


for i in range(1, 31):
    df6[f'lag{i}'] = df6['daily_columno3'].shift(i)


df6.dropna(inplace=True)
print(df6.shape)
df6.head()

(65480, 33)


,daily_date,daily_columno3,ema_avg,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag21,lag22,lag23,lag24,lag25,lag26,lag27,lag28,lag29,lag30
30,1976-09-07,273.8,287.181151,289.4,280.6,284.5,290.3,295.2,303.0,300.1,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
31,1976-09-08,284.5,280.490575,273.8,289.4,280.6,284.5,290.3,295.2,303.0,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
32,1976-09-09,283.5,282.495288,284.5,273.8,289.4,280.6,284.5,290.3,295.2,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
33,1976-09-10,283.5,282.997644,283.5,284.5,273.8,289.4,280.6,284.5,290.3,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
34,1976-09-13,307.9,283.248822,283.5,283.5,284.5,273.8,289.4,280.6,284.5,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [ ]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 31)]
lag_features.append('ema_avg')
X = df6[lag_features]
y = df6['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")



[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001183 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7905
[LightGBM] [Info] Number of data points in the train set: 52384, number of used features: 31
[LightGBM] [Info] Start training from score 0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.648
MAE : 0.472
R2  : 52.73 %

----- Decision Tree -----
RMSE: 0.935
MAE : 0.675
R2  : 1.64 %

----- Random Forest -----
RMSE: 0.649
MAE : 0.473
R2  : 52.64 %

----- Gradient Boosting -----
RMSE: 0.648
MAE : 0.472
R2  : 52.68 %

----- Support Vector Regressor -----
RMSE: 0.663
MAE : 0.477
R2  : 50.49 %

----- XGBoost -----
RMSE: 0.669
MAE : 0.482
R2  : 49.64 %

----- LightGBM -----
RMSE: 0.647
MAE : 0.471
R2  : 52.89 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


***Tuining 6**

In [ ]:

# Ensure data type
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

# Define models with GPU enabled where possible
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(tree_method='gpu_hist', predictor='gpu_predictor', random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(device='gpu', random_state=42)
}

# Define hyperparameter grids

param_grids = {
    'Linear Regression': {},
    'Decision Tree': {
        'max_depth': IntDistribution(5, 15),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Random Forest': {
        'n_estimators': IntDistribution(100, 150),
        'max_depth': IntDistribution(10, 20),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Gradient Boosting': {
        'n_estimators': IntDistribution(100, 150),
        'learning_rate': FloatDistribution(0.01, 0.1),
        'max_depth': IntDistribution(3, 5),
        'min_samples_split': IntDistribution(2, 5)
    },
    'Support Vector Regressor': {
        'C': FloatDistribution(0.1, 10),
        'kernel': CategoricalDistribution(['rbf', 'linear']),
        'epsilon': FloatDistribution(0.01, 0.1)
    },
    'XGBoost': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 7),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'gamma': FloatDistribution(0, 1),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(1, 2)
    },
    'LightGBM': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 5),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'num_leaves': IntDistribution(15, 31),
        'min_child_samples': IntDistribution(10, 20),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(0, 1)
    }
}

# Models to tune with OptunaSearchCV
models_to_optimize = {'Random Forest', 'Gradient Boosting', 'Support Vector Regressor', 'XGBoost', 'LightGBM'}

best_models = {}
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    if param_grids[name]:
        # Use OptunaSearchCV for tuning (Bayesian optimization)
        n_trials = 20
        search = OptunaSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            n_trials=n_trials,
            cv=3,
            scoring='r2',
            n_jobs=2,
            random_state=42,
            verbose=1
        )
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        print(f"{name} Best Parameters: {search.best_params_}")

    else:
        model.fit(X_train, y_train)
        best_models[name] = model

    # Predict and evaluate
    preds = best_models[name].predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'Model': best_models[name],
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

print("\nModel Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")

**Trial 7 : TrainTestSplit + lag_60 + rolling_std_7**

In [ ]:
# 2. Sort the dataframe by the full date (oldest to latest)
df7=df.copy()
df7 = df7.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df7['rolling_std'] = df7['daily_columno3'].shift(1).rolling(window=7).std()


for i in range(1, 61):
    df7[f'lag{i}'] = df7['daily_columno3'].shift(i)


df7.dropna(inplace=True)
print(df7.shape)
df7.head()

(65450, 63)


,daily_date,daily_columno3,rolling_std,lag1,lag2,lag3,lag4,lag5,lag6,lag7,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
60,1980-02-26,340.0,13.403820,286.0,287.4,297.2,311.8,294.2,267.9,296.2,...,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2,264.0
61,1980-02-27,308.0,22.845746,340.0,286.0,287.4,297.2,311.8,294.2,267.9,...,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2,257.2
62,1980-03-05,424.0,18.766231,308.0,340.0,286.0,287.4,297.2,311.8,294.2,...,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9,296.2
63,1980-03-06,323.0,48.539017,424.0,308.0,340.0,286.0,287.4,297.2,311.8,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,48.328696,323.0,424.0,308.0,340.0,286.0,287.4,297.2,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7


In [ ]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_std')
X = df7[lag_features]
y = df7['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002088 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15555
[LightGBM] [Info] Number of data points in the train set: 52360, number of used features: 61
[LightGBM] [Info] Start training from score -0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.647
MAE : 0.471
R2  : 52.83 %

----- Decision Tree -----
RMSE: 0.930
MAE : 0.672
R2  : 2.61 %

----- Random Forest -----
RMSE: 0.649
MAE : 0.473
R2  : 52.65 %

----- Gradient Boosting -----
RMSE: 0.652
MAE : 0.472
R2  : 52.17 %

----- Support Vector Regressor -----
RMSE: 0.669
MAE : 0.482
R2  : 49.68 %

----- XGBoost -----
RMSE: 0.672
MAE : 0.483
R2  : 49.25 %

----- LightGBM -----
RMSE: 0.649
MAE : 0.471
R2  : 52.55 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**Tuning 7**

In [ ]:

# Ensure data type
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

# Define models with GPU enabled where possible
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(tree_method='gpu_hist', predictor='gpu_predictor', random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(device='gpu', random_state=42)
}

# Define hyperparameter grids

param_grids = {
    'Linear Regression': {},
    'Decision Tree': {
        'max_depth': IntDistribution(5, 15),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Random Forest': {
        'n_estimators': IntDistribution(100, 150),
        'max_depth': IntDistribution(10, 20),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Gradient Boosting': {
        'n_estimators': IntDistribution(100, 150),
        'learning_rate': FloatDistribution(0.01, 0.1),
        'max_depth': IntDistribution(3, 5),
        'min_samples_split': IntDistribution(2, 5)
    },
    'Support Vector Regressor': {
        'C': FloatDistribution(0.1, 10),
        'kernel': CategoricalDistribution(['rbf', 'linear']),
        'epsilon': FloatDistribution(0.01, 0.1)
    },
    'XGBoost': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 7),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'gamma': FloatDistribution(0, 1),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(1, 2)
    },
    'LightGBM': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 5),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'num_leaves': IntDistribution(15, 31),
        'min_child_samples': IntDistribution(10, 20),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(0, 1)
    }
}

# Models to tune with OptunaSearchCV
models_to_optimize = {'Random Forest', 'Gradient Boosting', 'Support Vector Regressor', 'XGBoost', 'LightGBM'}

best_models = {}
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    if param_grids[name]:
        # Use OptunaSearchCV for tuning (Bayesian optimization)
        n_trials = 20
        search = OptunaSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            n_trials=n_trials,
            cv=3,
            scoring='r2',
            n_jobs=2,
            random_state=42,
            verbose=1
        )
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        print(f"{name} Best Parameters: {search.best_params_}")

    else:
        model.fit(X_train, y_train)
        best_models[name] = model

    # Predict and evaluate
    preds = best_models[name].predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'Model': best_models[name],
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

print("\nModel Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")

***Trial 8 : TrainTestSplit + lag_60 + rolling_avg_3 + rolling_std_3***

In [ ]:
# 2. Sort the dataframe by the full date (oldest to latest)
df8=df.copy()
df8 = df8.sort_values('daily_date', ascending=True).reset_index(drop=True)

# Rolling average feature (3-day) - To Avoid data leakage, we shift the column by 1 day
df8['rolling_std'] = df8['daily_columno3'].shift(1).rolling(window=3).std()
df8['rolling_avg'] = df5['daily_columno3'].shift(1).rolling(window=3).mean()


for i in range(1, 61):
    df8[f'lag{i}'] = df8['daily_columno3'].shift(i)


df8.dropna(inplace=True)
print(df8.shape)
df8.head()

(65447, 64)


,daily_date,daily_columno3,rolling_std,rolling_avg,lag1,lag2,lag3,lag4,lag5,lag6,...,lag51,lag52,lag53,lag54,lag55,lag56,lag57,lag58,lag59,lag60
63,1980-03-06,323.0,59.911045,357.333333,424.0,308.0,340.0,286.0,287.4,297.2,...,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7,308.9
64,1980-03-12,373.0,63.089883,351.666667,323.0,424.0,308.0,340.0,286.0,287.4,...,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8,351.7
65,1980-03-13,363.0,50.500825,373.333333,373.0,323.0,424.0,308.0,340.0,286.0,...,295.2,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8,347.8
66,1980-03-24,381.0,26.457513,353.000000,363.0,373.0,323.0,424.0,308.0,340.0,...,305.9,295.2,296.2,342.0,365.4,332.2,332.2,309.8,330.3,347.8
67,1980-03-25,333.0,9.018500,372.333333,381.0,363.0,373.0,323.0,424.0,308.0,...,307.9,305.9,295.2,296.2,342.0,365.4,332.2,332.2,309.8,330.3


In [ ]:
minmax_x = StandardScaler()
minmax_y= StandardScaler()
lag_features = [f'lag{i}' for i in range(1, 61)]
lag_features.append('rolling_std')
lag_features.append('rolling_avg')
X = df8[lag_features]
y = df8['daily_columno3']


# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
X_train = minmax_x.fit_transform(X_train)
y_train = minmax_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
X_test = minmax_x.transform(X_test)
y_test = minmax_y.transform(y_test.values.reshape(-1, 1)).flatten()

# STEP 3: Define models
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42)
}

# STEP 4: Train, Predict, and Evaluate
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    results[name] = {
        'Model': model,
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

# STEP 5: Print evaluation metrics
print("Model Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002578 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15810
[LightGBM] [Info] Number of data points in the train set: 52357, number of used features: 62
[LightGBM] [Info] Start training from score 0.000000
Model Evaluation Metrics:

----- Linear Regression -----
RMSE: 0.647
MAE : 0.470
R2  : 52.92 %

----- Decision Tree -----
RMSE: 0.927
MAE : 0.670
R2  : 3.27 %

----- Random Forest -----
RMSE: 0.650
MAE : 0.474
R2  : 52.44 %

----- Gradient Boosting -----
RMSE: 0.650
MAE : 0.471
R2  : 52.47 %

----- Support Vector Regressor -----
RMSE: 0.669
MAE : 0.482
R2  : 49.57 %

----- XGBoost -----
RMSE: 0.672
MAE : 0.481
R2  : 49.26 %

----- LightGBM -----
RMSE: 0.649
MAE : 0.471
R2  : 52.64 %



/home/t2420363/miniconda3/envs/rapids-23.12/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


**Tuning 8**

In [ ]:

# Ensure data type
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test = y_test.astype(np.float32)

# Define models with GPU enabled where possible
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR(),
    'XGBoost': XGBRegressor(tree_method='gpu_hist', predictor='gpu_predictor', random_state=42, verbosity=0),
    'LightGBM': LGBMRegressor(device='gpu', random_state=42)
}

# Define hyperparameter grids

param_grids = {
    'Linear Regression': {},
    'Decision Tree': {
        'max_depth': IntDistribution(5, 15),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Random Forest': {
        'n_estimators': IntDistribution(100, 150),
        'max_depth': IntDistribution(10, 20),
        'min_samples_split': IntDistribution(2, 5),
        'min_samples_leaf': IntDistribution(1, 2)
    },
    'Gradient Boosting': {
        'n_estimators': IntDistribution(100, 150),
        'learning_rate': FloatDistribution(0.01, 0.1),
        'max_depth': IntDistribution(3, 5),
        'min_samples_split': IntDistribution(2, 5)
    },
    'Support Vector Regressor': {
        'C': FloatDistribution(0.1, 10),
        'kernel': CategoricalDistribution(['rbf', 'linear']),
        'epsilon': FloatDistribution(0.01, 0.1)
    },
    'XGBoost': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 7),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'gamma': FloatDistribution(0, 1),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(1, 2)
    },
    'LightGBM': {
        'n_estimators': IntDistribution(100, 200),
        'max_depth': IntDistribution(3, 5),
        'learning_rate': FloatDistribution(0.01, 0.05),
        'num_leaves': IntDistribution(15, 31),
        'min_child_samples': IntDistribution(10, 20),
        'subsample': FloatDistribution(0.6, 0.8),
        'colsample_bytree': FloatDistribution(0.6, 0.8),
        'reg_alpha': FloatDistribution(0, 0.1),
        'reg_lambda': FloatDistribution(0, 1)
    }
}

# Models to tune with OptunaSearchCV
models_to_optimize = {'Random Forest', 'Gradient Boosting', 'Support Vector Regressor', 'XGBoost', 'LightGBM'}

best_models = {}
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    if param_grids[name]:
        # Use OptunaSearchCV for tuning (Bayesian optimization)
        n_trials = 20
        search = OptunaSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            n_trials=n_trials,
            cv=3,
            scoring='r2',
            n_jobs=2,
            random_state=42,
            verbose=1
        )
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        print(f"{name} Best Parameters: {search.best_params_}")

    else:
        model.fit(X_train, y_train)
        best_models[name] = model

    # Predict and evaluate
    preds = best_models[name].predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results[name] = {
        'Model': best_models[name],
        'Predictions': preds,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2
    }

print("\nModel Evaluation Metrics:\n")
for name, res in results.items():
    print(f"----- {name} -----")
    print(f"RMSE: {res['RMSE']:.3f}")
    print(f"MAE : {res['MAE']:.3f}")
    print(f"R2  : {(res['R2']*100):.2f} %\n")